# Fine-tuning Gemma 4 for Khmer and Lao OCR with Unsloth

This notebook fine-tunes a vision-language model to transcribe Khmer or Lao text from images. Khmer is the default; set `OCR_LANGUAGE=lao` before starting Jupyter to use the paired Lao document dataset. Training requires a CUDA-capable GPU and a CUDA-enabled PyTorch/Unsloth environment.

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation and environment check

Use a GPU runtime. On this project's Windows setup, use the dedicated `.venv-unsloth` kernel described in the repository README. The regular `.venv` has CPU-only PyTorch and is intended for the other course notebooks.

In [1]:
%%capture
import importlib.metadata as metadata
import logging
import os, re, sys
from pathlib import Path

# Keep model caches in the project. This avoids a Windows startup hang when
# Unsloth probes a protected or redirected user cache directory.
default_hf_home = (
    Path("D:/huggingface-cache/llm-fundamentals-tp2")
    if Path("D:/").exists()
    else Path.cwd() / ".cache" / "huggingface"
)
hf_home = Path(os.environ.get("HF_HOME", default_hf_home))
hf_hub_cache = Path(os.environ.get("HF_HUB_CACHE", hf_home / "hub"))

hf_xet_cache = hf_home / "xet"
hf_hub_cache.mkdir(parents=True, exist_ok=True)
hf_xet_cache.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(hf_home)
os.environ["HF_HUB_CACHE"] = str(hf_hub_cache)
os.environ["HF_XET_CACHE"] = str(hf_xet_cache)
# Xet repeatedly stalls on some Windows/network combinations. Force
# resumable HTTPS with longer timeouts before importing Hugging Face.
os.environ["UNSLOTH_STABLE_DOWNLOADS"] = "0"
os.environ["UNSLOTH_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"

if "COLAB_" not in "".join(os.environ.keys()):
    try:
        metadata.version("unsloth")
    except metadata.PackageNotFoundError as exc:
        raise RuntimeError(
            f"Unsloth is missing from the selected kernel: {sys.executable}\n"
            "Close Jupyter, run .\\scripts\\start_ocr.cmd from the project folder, "
            "then restart the kernel and run all cells."
        ) from exc
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
    !pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
    !pip install "huggingface_hub>=1.5.0,<2.0"

try:
    torch_version = tuple(int(x) for x in metadata.version("torch").split("+")[0].split(".")[:2])
except Exception:
    torch_version = (2, 11)
torchcodec = {(2, 11): "torchcodec>=0.11.0", (2, 10): "torchcodec==0.10.*",
              (2, 9): "torchcodec==0.9.*", (2, 8): "torchcodec==0.7.*",
              (2, 7): "torchcodec==0.5.*"}.get(
                  torch_version, "torchcodec>=0.12.0" if torch_version >= (2, 12) else "torchcodec"
              )
if "COLAB_" in "".join(os.environ.keys()):
    !pip install "{torchcodec}"

import torch
torch._dynamo.config.recompile_limit = 64

# Unsloth currently registers two Transformer Engine enums that modern
# PyTorch already supports. Hide only that known deprecation warning.
logging.getLogger("torch.utils._pytree").setLevel(logging.ERROR)

RuntimeError: Unsloth is missing from the selected kernel: c:\Users\USER\Documents\Self Study\AI\Machine Learning Bootcamp\env\Scripts\python.exe
Close Jupyter, run .\scripts\start_ocr.cmd from the project folder, then restart the kernel and run all cells.

In [2]:
%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

### Unsloth

In [3]:
import os
from pathlib import Path

# Always bypass Xet here, even when this cell is run without the setup cell.
# The Windows Xet transport stalls on this machine; HTTPS is resumable.
os.environ["UNSLOTH_STABLE_DOWNLOADS"] = "0"
os.environ["UNSLOTH_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "600"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
model_cache = Path(os.environ.get(
    "HF_HUB_CACHE",
    "D:/huggingface-cache/llm-fundamentals-tp2/hub"
    if Path("D:/").exists()
    else str(Path.cwd() / ".cache" / "huggingface" / "hub"),
))
model_cache.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(model_cache)

# huggingface_hub caches this value when imported. Update an existing
# import too, so rerunning this cell in the same kernel still works.
try:
    import huggingface_hub.constants as hf_constants
    hf_constants.HF_HUB_DISABLE_XET = True
except ImportError:
    pass

import torch
from unsloth import FastVisionModel

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Select the project's .venv-unsloth kernel or use Google Colab with a GPU runtime."
    )

gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
] # More models at https://huggingface.co/unsloth

default_local_model = (
    Path("D:/models/gemma-4-E2B-it-unsloth-bnb-4bit")
    if Path("D:/").exists()
    else Path.cwd() / ".models" / "gemma-4-E2B-it-unsloth-bnb-4bit"
)
LOCAL_MODEL_DIR = Path(os.getenv("UNSLOTH_LOCAL_MODEL", default_local_model))
EXPECTED_SHARDS = {
    "model-00001-of-00003.safetensors": 1_422_130_776,
    "model-00002-of-00003.safetensors": 4_986_348_677,
    "model-00003-of-00003.safetensors": 1_716_554_394,
}
local_model_ready = (LOCAL_MODEL_DIR / "config.json").is_file() and all(
    (LOCAL_MODEL_DIR / name).is_file()
    and (LOCAL_MODEL_DIR / name).stat().st_size == expected_size
    for name, expected_size in EXPECTED_SHARDS.items()
)
MODEL_NAME = os.getenv(
    "UNSLOTH_MODEL",
    str(LOCAL_MODEL_DIR) if local_model_ready else "unsloth/gemma-4-E2B-it",
)
model, processor = FastVisionModel.from_pretrained(
    MODEL_NAME,
    cache_dir = str(model_cache),
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

ModuleNotFoundError: No module named 'unsloth'

We now add LoRA adapters for parameter efficient fine-tuning, allowing us to train only 1% of all model parameters efficiently.

**[NEW]** We also support fine-tuning only the vision component, only the language component, or both. Additionally, you can choose to fine-tune the attention modules, the MLP layers, or both!

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,                           # Conservative setting for an 8 GB GPU
    lora_alpha = 16,                  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,               # We support rank stabilized LoRA
    loftq_config = None,               # And LoftQ
    target_modules = "all-linear",    # Optional now! Can specify a list if needed
)

<a name="Data"></a>
### Data Prep
Choose `khmer` (default) or `lao` with the `OCR_LANGUAGE` environment variable. The Khmer dataset is [KiteAether/c_khmer_gemma_ocr_train](https://huggingface.co/datasets/KiteAether/c_khmer_gemma_ocr_train). The Lao loader pairs the `.png` and `.gt.txt` files in [Monaliza14xx/lao-documents](https://huggingface.co/datasets/Monaliza14xx/lao-documents).

The Lao repository does not currently declare a license in its dataset card; confirm that you have permission before redistributing trained weights.

In [ ]:
import re
from pathlib import Path

from datasets import Dataset, Image as DatasetImage, load_dataset
from huggingface_hub import snapshot_download

OCR_LANGUAGE = os.getenv("OCR_LANGUAGE", "khmer").strip().lower()
LANGUAGE_LABELS = {"khmer": "Khmer", "lao": "Lao"}
if OCR_LANGUAGE not in LANGUAGE_LABELS:
    raise ValueError("OCR_LANGUAGE must be 'khmer' or 'lao'.")

if OCR_LANGUAGE == "khmer":
    DATASET_ID = "KiteAether/c_khmer_gemma_ocr_train"
    dataset = load_dataset(DATASET_ID, split="train")
else:
    DATASET_ID = "Monaliza14xx/lao-documents"
    repo_dir = Path(snapshot_download(
        repo_id=DATASET_ID, repo_type="dataset",
        allow_patterns=["*.png", "*.gt.txt"],
    ))

    def natural_key(path):
        return [int(part) if part.isdigit() else part.lower()
                for part in re.split(r"(\d+)", path.name)]

    image_paths = sorted(repo_dir.rglob("*.png"), key=natural_key)
    records = []
    for image_path in image_paths:
        text_path = image_path.with_name(f"{image_path.stem}.gt.txt")
        if text_path.exists():
            records.append({
                "image": str(image_path),
                "text": text_path.read_text(encoding="utf-8").strip(),
            })
    if not records:
        raise RuntimeError("No paired Lao .png/.gt.txt files were found.")
    dataset = Dataset.from_list(records).cast_column("image", DatasetImage())

required_columns = {"image", "text"}
if not required_columns.issubset(dataset.column_names):
    raise ValueError(f"Dataset must contain {required_columns}; found {set(dataset.column_names)}")
dataset = dataset.filter(lambda row: bool(row["text"].strip()))
LANGUAGE_LABEL = LANGUAGE_LABELS[OCR_LANGUAGE]
instruction = f"Extract the {LANGUAGE_LABEL} text from this image. Return only the transcription."
print(f"Loaded {len(dataset):,} {LANGUAGE_LABEL} OCR examples from {DATASET_ID}.")

Inspect an image and its target transcription before training.

In [ ]:
dataset

In [ ]:
dataset[2]["image"]

In [ ]:
dataset[2]["text"]

Print another target transcription to verify Unicode rendering:

In [ ]:
target_text = dataset[3]["text"]
print(target_text)

Format every example as one user message with the OCR instruction and image, followed by one assistant message with the ground-truth transcription:

```python
[
    {
        "role": "user",
        "content": [
            {"type": "text", "text": instruction},
            {"type": "image", "image": sample["image"]},
        ],
    },
    {
        "role": "assistant",
        "content": [
            {"type": "text", "text": sample["text"]},
        ],
    },
]
```

In [ ]:
def convert_to_conversation(sample):
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": instruction},
                {"type": "image", "image": sample["image"]},
            ],
        },
        {"role": "assistant", "content": [{"type": "text", "text": sample["text"]}]},
    ]
    return {"messages": conversation}


Let's convert the dataset into the "correct" format for finetuning:

In [ ]:
splits = dataset.train_test_split(test_size=0.1, seed=3407)
train_dataset = splits["train"]
eval_dataset = splits["test"]
converted_dataset = [convert_to_conversation(sample) for sample in train_dataset]
converted_eval_dataset = [convert_to_conversation(sample) for sample in eval_dataset]
print(f"Training examples: {len(converted_dataset):,}")
print(f"Held-out examples: {len(converted_eval_dataset):,}")

The first example is now structured like below:

In [ ]:
converted_dataset[0]

Let's inspect how the first training conversation is structured:

In [ ]:
from unsloth import get_chat_template

processor = get_chat_template(
    processor,
    "gemma-4"
)

Before fine-tuning, let us evaluate the base model's performance. We do not expect strong results, as it has not encountered this chat template before.

In [ ]:
FastVisionModel.for_inference(model)
image = eval_dataset[0]["image"]

messages = [
    {
        "role": "user",
        "content": [{"type": "image"}, {"type": "text", "text": instruction}],
    }
]
input_text = processor.apply_chat_template(messages, add_generation_prompt = True)
inputs = processor(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer

text_streamer = TextStreamer(processor, skip_prompt = True)
result = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                        use_cache = True, temperature = 1.0, top_p = 0.95, top_k = 64)

You can see it's absolutely terrible! It doesn't follow instructions at all

<a name="Train"></a>
### Train the model
The defaults are conservative for an 8 GB laptop GPU: micro-batch 1, gradient accumulation 4, and a bounded 60-step smoke run. Increase `MAX_STEPS` only after the first run is stable.

We use our new `UnslothVisionDataCollator` which will help in our vision finetuning setup.

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

MAX_STEPS = int(os.getenv("MAX_STEPS", "60"))

trainer = SFTTrainer(
    model = model,
    train_dataset = converted_dataset,
    processing_class = processor.tokenizer,
    data_collator = UnslothVisionDataCollator(model, processor),
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        max_grad_norm = 0.3,
        warmup_steps = 2,
        max_steps = MAX_STEPS,
        learning_rate = 2e-4,
        logging_steps = 1,
        save_strategy = "steps",
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # For Weights and Biases or others

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 1024,
        dataloader_num_workers = 0,
    )
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
FastVisionModel.for_training(model)
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Run the fine-tuned model on a held-out example. The instruction is selected automatically for Khmer or Lao.

We'll use the best hyperparameters for inference on Gemma: `top_p=0.95`, `top_k=64`, and `temperature=1.0`.

In [ ]:
FastVisionModel.for_inference(model)
image = eval_dataset[min(1, len(eval_dataset) - 1)]["image"]

messages = [
    {
        "role": "user",
        "content": [{"type": "image"}, {"type": "text", "text": instruction}],
    }
]

input_text = processor.apply_chat_template(messages, add_generation_prompt = True)

inputs = processor(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer

text_streamer = TextStreamer(processor, skip_prompt = True)
result = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                        use_cache = True, temperature = 0.01, top_p = 0.95, top_k = 1)

In [ ]:
image

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, use Hugging Face’s `push_to_hub` for online saving, or `save_pretrained` for local storage.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
ARTIFACTS_DIR = Path(os.getenv("PROJECT_ROOT", Path.cwd())) / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = str(ARTIFACTS_DIR / f"gemma_4_{OCR_LANGUAGE}_ocr_lora")
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Saved adapter and processor to {OUTPUT_DIR}")

To reload the saved LoRA adapter for inference, set `LOAD_SAVED_ADAPTER=1` before starting Jupyter:

In [ ]:
LOAD_SAVED_ADAPTER = os.getenv("LOAD_SAVED_ADAPTER", "0") == "1"
if LOAD_SAVED_ADAPTER:
    from unsloth import FastVisionModel

    model, processor = FastVisionModel.from_pretrained(
        model_name = OUTPUT_DIR,
        load_in_4bit = True,  # Set to False for 16bit LoRA
    )

FastVisionModel.for_inference(model)
sample = eval_dataset[0]
image = sample["image"].convert("RGB")
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": instruction,
            },
            {
                "type": "image",
            },
        ],
    },
]
input_text = processor.apply_chat_template(messages, add_generation_prompt = True)
inputs = processor(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer

text_streamer = TextStreamer(processor.tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.0, top_p = 0.95, top_k = 64)

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Enable exactly one export after training. Merged 16-bit weights need substantial disk space.
EXPORT_MERGED_16BIT = False
PUSH_MERGED_TO_HUB = False
MERGED_DIR = str(ARTIFACTS_DIR / f"gemma_4_{OCR_LANGUAGE}_ocr_merged_16bit")

if EXPORT_MERGED_16BIT:
    model.save_pretrained_merged(MERGED_DIR, processor, save_method = "merged_16bit")

if PUSH_MERGED_TO_HUB:
    model.push_to_hub_merged(
        f"YOUR_USERNAME/{Path(MERGED_DIR).name}", processor,
        save_method = "merged_16bit", token = "YOUR_HF_TOKEN",
    )

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).